In [1]:
import polib
import requests
import time
import urllib.parse
import re
from concurrent.futures import ThreadPoolExecutor, as_completed


INPUT_PO = r"C:\Users\xx\xx\Desktop\mo2\EN.po"
OUTPUT_PO = r"C:\Users\xxx\xx\Desktop\mo2\TR_FULL.po"
OUTPUT_MO = r"C:\Users\xx\xxx\Desktop\mo2\TR_FULL.mo"


MAX_WORKERS = 20
REQUEST_TIMEOUT = 6
DELAY_BETWEEN_REQUESTS = 0.05


def should_skip_translation(text):
    
    if not text or not isinstance(text, str) or not text.strip():
        return True

    
    if re.match(r'^@+$', text.strip()):
        return True

    
    if re.match(r'^<<m:\{.*?\}>>$', text.strip()):
        return True

    
    if re.match(r'^\[\{.*?\}\]$', text.strip()) or re.match(r'^\{\[.*?\}\]$', text.strip()) or re.match(r'^\{.*?\}$', text.strip()):
        return True

    
    if len(text.strip()) == 1 and text.strip() in ['#', '$', '%', '&', '*', '-', '+', '=', '/', '\\']:
        return True

    return False


def translate_google_simple(text):
    
    if should_skip_translation(text):
        return text

    try:
        url = "https://translate.googleapis.com/translate_a/single"
        params = {
            'client': 'gtx',
            'sl': 'en',
            'tl': 'tr',
            'dt': 't',
            'q': text
        }

        response = requests.get(url, params=params, timeout=REQUEST_TIMEOUT)

        if response.status_code == 200:
            data = response.json()
            if data and data[0]:
                translated = data[0][0][0]
                if translated and translated != text:
                    return translated
        return text

    except Exception:
        return text


def split_keep_tokens(text):
    
    pattern = r'(\{.*?\}|\[\{.*?\}\]|<<m:.*?>>|<<m>>|<<m:.*?$|<.*?>)'
    parts = re.split(pattern, text)
    return [p for p in parts if p]  # boşları at


def smart_translate(text):
    
    if should_skip_translation(text):
        return text

    parts = split_keep_tokens(text)
    translated_parts = []

    for part in parts:
        # Eğer kod parçası veya HTML tag’iyse dokunma
        if re.match(r'^(\{.*?\}|\[\{.*?\}\]|<<m:.*?>>|<<m>>|<<m:.*?$|<.*?>)$', part.strip()):
            translated_parts.append(part)
        else:
            # Normal metin → Google'a gönder
            translated_parts.append(translate_google_simple(part))

    return "".join(translated_parts)


def process_batch(batch):
    
    results = []
    for text in batch:
        results.append(smart_translate(text))
        time.sleep(DELAY_BETWEEN_REQUESTS)
    return results


def mass_translate():
    
    print("📦 PO dosyası yükleniyor...")
    po = polib.pofile(INPUT_PO)
    print(f"✅ PO dosyası yüklendi: {len(po)} entries")

    # TÜM METİNLERİ İŞLE
    texts_to_translate = []
    entries_to_update = []
    skipped_count = 0

    for i, entry in enumerate(po):
        if should_skip_translation(entry.msgstr):
            skipped_count += 1
        else:
            if entry.msgstr.strip() and entry.msgstr != entry.msgid:
                texts_to_translate.append(entry.msgstr)
                entries_to_update.append((i, entry))
            else:
                skipped_count += 1

    print(f"🎯 Çevrilecek {len(texts_to_translate)} metin bulundu")
    print(f"⏭️  Atlanacak {skipped_count} metin (sadece özel durumlar)")
    print("🚀 100'ER 100'ER çeviri başlıyor...\n")

    start_time = time.time()
    total_translated = 0
    batch_size = 100 

    
    for batch_start in range(0, len(texts_to_translate), batch_size):
        batch_end = min(batch_start + batch_size, len(texts_to_translate))
        current_batch = texts_to_translate[batch_start:batch_end]
        current_entries = entries_to_update[batch_start:batch_end]

        print(f"🔄 {batch_start}-{batch_end} arası {len(current_batch)} metin işleniyor...")

        
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            future_to_index = {
                executor.submit(smart_translate, text): idx
                for idx, text in enumerate(current_batch)
            }

            for future in as_completed(future_to_index):
                idx = future_to_index[future]
                try:
                    translated_text = future.result()
                    original_idx, entry = current_entries[idx]

                    if translated_text != entry.msgstr:
                        entry.msgstr = translated_text
                        total_translated += 1

                except Exception:
                    pass

        
        elapsed = time.time() - start_time
        progress_percent = (batch_end / len(texts_to_translate)) * 100
        speed = total_translated / elapsed if elapsed > 0 else 0

        print(f"✅ {batch_end}/{len(texts_to_translate)} ({progress_percent:.1f}%) - "
              f"Çevrilen: {total_translated} - Hız: {speed:.1f} çeviri/sn - Süre: {elapsed:.1f}s\n")

        
        if batch_end % 1000 == 0:
            po.save(OUTPUT_PO)
            print(f"💾 Checkpoint: {batch_end}. çeviri kaydedildi")

    
    po.save(OUTPUT_PO)
    po.save_as_mofile(OUTPUT_MO)

    total_time = time.time() - start_time
    print(f"\n🎉 ÇEVİRİ TAMAMLANDI!")
    print(f"📊 Toplam çeviri: {total_translated}")
    print(f"⏭️  Toplam atlanan: {skipped_count}")
    print(f"⏱️  Toplam süre: {total_time:.1f} saniye")
    print(f"🚀 Ortalama hız: {total_translated/total_time:.1f} çeviri/saniye")
    print(f"💾 Dosyalar kaydedildi: {OUTPUT_PO}")



if __name__ == "__main__":
    mass_translate()


📦 PO dosyası yükleniyor...
✅ PO dosyası yüklendi: 40635 entries
🎯 Çevrilecek 40381 metin bulundu
⏭️  Atlanacak 254 metin (sadece özel durumlar)
🚀 100'ER 100'ER çeviri başlıyor...

🔄 0-100 arası 100 metin işleniyor...
✅ 100/40381 (0.2%) - Çevrilen: 95 - Hız: 8.7 çeviri/sn - Süre: 10.9s

🔄 100-200 arası 100 metin işleniyor...
✅ 200/40381 (0.5%) - Çevrilen: 193 - Hız: 8.8 çeviri/sn - Süre: 21.8s

🔄 200-300 arası 100 metin işleniyor...
✅ 300/40381 (0.7%) - Çevrilen: 293 - Hız: 8.4 çeviri/sn - Süre: 34.9s

🔄 300-400 arası 100 metin işleniyor...
✅ 400/40381 (1.0%) - Çevrilen: 392 - Hız: 5.9 çeviri/sn - Süre: 66.6s

🔄 400-500 arası 100 metin işleniyor...
✅ 500/40381 (1.2%) - Çevrilen: 472 - Hız: 6.0 çeviri/sn - Süre: 78.9s

🔄 500-600 arası 100 metin işleniyor...
✅ 600/40381 (1.5%) - Çevrilen: 557 - Hız: 6.1 çeviri/sn - Süre: 91.2s

🔄 600-700 arası 100 metin işleniyor...
✅ 700/40381 (1.7%) - Çevrilen: 626 - Hız: 6.0 çeviri/sn - Süre: 104.6s

🔄 700-800 arası 100 metin işleniyor...
✅ 800/40381 (